In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_silver/11_silver_careplans
# Purpose  : Not currently used in Gold (none of the 14 business questions
#            require it), but processed in Silver so it is immediately
#            available if fact_careplans is needed later.
# ============================================================

import dlt
from pyspark.sql import functions as F

TEMPORAL_VALID_RULE_DATE = "stop_date IS NULL OR stop_date >= start_date"


@dlt.table(name="covid19_lakehouse.silver.careplans", comment="Careplans with correct types")
@dlt.expect_all_or_drop({
    "valid_patient_id": "patient_id IS NOT NULL",
    "valid_encounter_id": "encounter_id IS NOT NULL",
})
@dlt.expect_or_drop("valid_stop_after_start", TEMPORAL_VALID_RULE_DATE)
def silver_careplans():
    df = dlt.read_stream("covid19_lakehouse.bronze.careplans")
    return (
        df.select(
            F.col("Id").alias("careplan_id"),
            F.col("PATIENT").alias("patient_id"),
            F.col("ENCOUNTER").alias("encounter_id"),
            F.col("CODE").alias("careplan_code"),
            F.col("DESCRIPTION").alias("careplan_description"),
            F.to_date("START").alias("start_date"),
            F.to_date("STOP").alias("stop_date"),
            F.when(F.col("STOP").isNull(), True).otherwise(False).alias("is_active"),
            F.col("REASONCODE").alias("reason_code"),
            F.col("REASONDESCRIPTION").alias("reason_description"),
        )
    )


@dlt.table(name="covid19_lakehouse.quarantine.careplans", comment="Careplans rejected by Silver DQ rules")
def quarantine_careplans():
    df = dlt.read_stream("covid19_lakehouse.bronze.careplans")
    return (
        df.filter(
            F.col("PATIENT").isNull()
            | F.col("ENCOUNTER").isNull()
            | (F.col("STOP").isNotNull() & (F.to_date("STOP") < F.to_date("START")))
        )
        .select(
            F.lit("careplans").alias("source_table"),
            F.lit("null_fk / stop_before_start").alias("rule_violated"),
            F.col("PATIENT").alias("patient_id"),
            F.col("ENCOUNTER").alias("encounter_id"),
            F.col("START").alias("raw_start"),
            F.col("STOP").alias("raw_stop"),
            F.current_timestamp().alias("quarantined_at"),
        )
    )